# Notebook 01 — EDA & Batch Layer (PEMS-BAY)
**Proyek:** Sistem Pemantauan & Prediksi Kemacetan Lalu Lintas Kota berbasis IoT + Big Data (Lambda Architecture)

## Posisi notebook ini di Lambda Architecture
```
            ┌──────────────── BATCH LAYER (notebook ini) ────────────────┐
 Sensor IoT │  Master dataset (immutable, Parquet)  →  Batch views (Spark) │──┐
 (PEMS-BAY) └─────────────────────────────────────────────────────────────┘  │   SERVING LAYER
                                                                             ├──► API + Dashboard
 Video CCTV ┌──────────────── SPEED LAYER (notebook 03 & 05) ─────────────┐  │   city planner
            │  YOLO + tracking → fitur real-time → Model A (nowcast)      │──┘
            └─────────────────────────────────────────────────────────────┘
```

## Yang akan kamu kuasai di notebook ini
1. Membaca & **memeriksa kualitas** data sensor nyata (bukan sintetis).
2. **Membersihkan** data deret waktu (timestamp hilang, sensor error).
3. Membuat label kemacetan yang **bermakna fisik**: rasio kecepatan terhadap *free-flow speed*.
4. Menyimpan **master dataset** dalam format Parquet terpartisi (prinsip Lambda: data mentah tidak pernah diubah).
5. Menghitung **batch views** dengan **PySpark** — tabel ringkasan yang nanti dibaca dashboard.

> Cara belajar yang saya sarankan: jalankan sel satu per satu, baca penjelasannya, lalu **tebak dulu hasilnya** sebelum melihat output. Di akhir ada 3 latihan — kerjakan sendiri, itu bagian terpenting.

## 0. Setup
Sel ini memasang **PySpark** (dan Java bila belum ada). Spark berjalan di atas JVM, jadi Java wajib ada.

In [ ]:
import sys, os, shutil
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and shutil.which("java") is None:
    !apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
%pip install -q pyspark==3.5.5 pyarrow     # pesan "restart the kernel" boleh diabaikan
print("Colab:", IN_COLAB, "| Java:", shutil.which("java"))

### Lokasi file
Notebook ini mengasumsikan struktur Google Drive seperti ini:
```
MyDrive/
└── Projek Akhir/
    └── Dataset/
        ├── PEMS-BAY.csv
        └── PEMS-BAY-META.csv
```
Kalau nama foldermu berbeda, **ubah variabel `BASE`** di bawah.

**Kenapa ada `WORK` terpisah?** Google Drive di Colab lambat untuk ribuan file kecil yang ditulis Spark. Jadi kita bekerja di disk lokal Colab (`/content/work`), lalu **menyalin hasil akhir** ke Drive.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/Projek Akhir"   # <-- sesuaikan bila perlu
    WORK = "/content/work"
else:
    BASE = os.environ.get("PROJ_BASE", ".")
    WORK = os.environ.get("PROJ_WORK", "./work")

DATA_DIR   = f"{BASE}/Dataset"
OUT_DIR    = f"{BASE}/output/01_batch"
MASTER_DIR = f"{WORK}/master_pemsbay"
for p in [WORK, OUT_DIR, f"{OUT_DIR}/figures"]:
    os.makedirs(p, exist_ok=True)

assert os.path.exists(f"{DATA_DIR}/PEMS-BAY.csv"), f"PEMS-BAY.csv tidak ditemukan di {DATA_DIR} — cek variabel BASE"
print("Data  :", DATA_DIR)
print("Output:", OUT_DIR)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyarrow as pa, pyarrow.parquet as pq

pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def savefig(name):
    # simpan setiap grafik — nanti langsung dipakai di laporan
    plt.savefig(f"{OUT_DIR}/figures/{name}.png", bbox_inches="tight")

## 1. Memahami data mentah
**PEMS-BAY** = kecepatan rata-rata (mil/jam) dari **325 sensor loop detector** di jalan tol Bay Area, California, tiap **5 menit**, Januari–Juni 2017. Sumber: Caltrans PeMS (dipakai di paper DCRNN, ICLR 2018). Lisensi file Zenodo: CC BY 4.0.

Format file-nya **wide**: baris = waktu, kolom = sensor.

In [ ]:
raw = pd.read_csv(f"{DATA_DIR}/PEMS-BAY.csv", index_col=0, parse_dates=True)
raw.columns = raw.columns.astype(int)
raw = raw.astype("float32")          # float32 = setengah memori float64, presisi cukup untuk kecepatan

print("Ukuran  :", raw.shape, "(baris = langkah waktu, kolom = sensor)")
print("Rentang :", raw.index.min(), "→", raw.index.max())
print("Total pembacaan:", f"{raw.size:,}")
raw.iloc[:5, :6]

**Metadata sensor** memberi lokasi (lat/lon), nomor jalan tol (`fwy`), arah, dan jumlah lajur. Ini yang nanti membuat peta di dashboard.

In [ ]:
meta = pd.read_csv(f"{DATA_DIR}/PEMS-BAY-META.csv")
meta = (meta[["sensor_id", "Fwy", "Dir", "County", "Latitude", "Longitude", "Lanes", "Type", "Name"]]
        .rename(columns=str.lower))
print(len(meta), "sensor di metadata")
print("Semua sensor di data punya metadata?", set(raw.columns) <= set(meta.sensor_id))
meta.head()

### 1.1 Cek kualitas data
Data sensor nyata **selalu** punya masalah. Tiga yang kita cek:
- **Timestamp hilang** — sistem tidak merekam pada jam tertentu.
- **NaN** — nilai kosong.
- **Kecepatan 0** — di jalan tol, kecepatan 0 rata-rata 5 menit hampir pasti **sensor error**, bukan macet total.

In [ ]:
full_idx = pd.date_range(raw.index.min(), raw.index.max(), freq="5min")

# Jam yang TIDAK PERNAH ADA karena pergantian ke jam musim panas (DST) di California
n = len(full_idx)
nonexistent = (pd.Series(full_idx).dt.tz_localize("America/Los_Angeles", nonexistent="NaT",
                                                  ambiguous=np.ones(n, bool)).isna().values)
dst_gap  = full_idx[nonexistent]
full_idx = full_idx[~nonexistent]
missing_ts = full_idx.difference(raw.index)
n_zero = int((raw == 0).sum().sum()); n_nan = int(raw.isna().sum().sum())

print(f"Timestamp tidak tercatat     : {len(dst_gap) + len(missing_ts)}")
print(f"  ↳ jam DST (memang tidak ada): {len(dst_gap)}  ({dst_gap.min()} s.d. {dst_gap.max()})")
print(f"  ↳ benar-benar hilang        : {len(missing_ts)}")
print(f"Nilai NaN                    : {n_nan}")
print(f"Nilai 0 (sensor error)       : {n_zero} ({n_zero / raw.size:.4%})")

speeds = pd.Series(raw.values.ravel())
speeds[speeds > 0].describe().round(1)

**Pelajaran menarik:** semua timestamp yang "hilang" jatuh pada **12 Maret 2017 pukul 02:00–02:55**. Itu bukan sensor rusak — pada tanggal itu California memajukan jam dari 02:00 langsung ke 03:00 (*daylight saving time*). Jam tersebut **memang tidak pernah ada**, jadi tidak boleh kita isi dengan nilai tebakan.

> Moral data engineering: sebelum "memperbaiki" data, cari tahu **kenapa** data itu terlihat salah.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(speeds[speeds > 0], bins=80, color="#4C78A8")
ax.set(title="Distribusi kecepatan semua sensor", xlabel="Kecepatan (mph)", ylabel="Jumlah pembacaan")
savefig("01_distribusi_kecepatan"); plt.show()

**Bacaan grafik:** sebagian besar pembacaan menumpuk di 60–70 mph (lancar). Ekor kiri yang tipis = kemacetan. Artinya: **kondisi macet itu jarang** — ini akan jadi tantangan utama modeling (kelas tidak seimbang).

## 2. Pembersihan data
Tiga langkah, urutannya penting:
1. **Lengkapi timestamp** yang hilang → baris baru berisi NaN (jam DST tidak ikut, karena memang tidak ada).
2. **Ubah 0 menjadi NaN** — kita jujur bahwa nilainya tidak diketahui.
3. **Interpolasi celah pendek** (maksimal 3 langkah = 15 menit). Celah lebih panjang dibiarkan kosong, karena menebak 1 jam lalu lintas itu berisiko.

Kita juga menyimpan **flag `is_imputed`** supaya selalu tahu nilai mana yang hasil tebakan. Ini praktik data engineering yang baik: *jangan pernah menyembunyikan imputasi*.

In [ ]:
df = raw.reindex(full_idx)                       # 1) lengkapi timestamp
df = df.mask(df == 0)                            # 2) 0 → NaN
was_missing = df.isna()
df = df.interpolate(method="time", limit=3, limit_area="inside")   # 3) isi celah ≤ 15 menit
is_imputed = was_missing & df.notna()

print("Nilai diisi interpolasi :", int(is_imputed.sum().sum()))
print("Nilai masih kosong      :", int(df.isna().sum().sum()))
del raw, speeds

## 3. Label kemacetan dari *free-flow speed*
Kecepatan absolut menyesatkan: 45 mph di jalan tol = melambat, tapi 45 km/jam di jalan kota = lancar. Solusinya, bandingkan dengan **kecepatan arus bebas (free-flow speed, v_ff)** sensor itu sendiri:

$$\text{speed\_ratio} = \frac{v}{v_{ff}}$$

**v_ff** = persentil ke-85 kecepatan pada jam sepi (22.00–05.00). Persentil ke-85 adalah konvensi umum rekayasa lalu lintas untuk kecepatan "normal" pengemudi.

Lalu kita bagi menjadi 4 level (ambang ini **keputusan desain proyek** — tulis alasannya di laporan):

| Level | speed_ratio | Arti |
|---|---|---|
| Low | ≥ 0,8 | lancar |
| Moderate | 0,6 – 0,8 | melambat |
| High | 0,4 – 0,6 | padat |
| Severe | < 0,4 | macet berat / stop-and-go |

> ⚠️ **Catatan kebocoran data:** di sini v_ff dihitung dari 6 bulan penuh — aman untuk *analisis* batch. Di notebook 04 (model forecast), v_ff **wajib** dihitung hanya dari periode training.

In [ ]:
night = df[(df.index.hour >= 22) | (df.index.hour < 5)]
v_ff = night.quantile(0.85).rename("v_ff")
v_ff.index.name = "sensor_id"
ratio = (df / v_ff).clip(upper=1.5)

BINS   = [0, 0.4, 0.6, 0.8, np.inf]
LEVELS = ["Severe", "High", "Moderate", "Low"]

r = ratio.values.ravel(); r = r[~np.isnan(r)]
dist = pd.Series(pd.cut(r, BINS, labels=LEVELS, right=False)).value_counts(normalize=True).reindex(LEVELS[::-1])
print("v_ff (mph):", v_ff.describe().round(1).to_dict())
print("\nDistribusi level kemacetan:")
for lv in dist.index: print(f"  {lv:9s} {dist[lv] * 100:6.2f} %")

In [ ]:
colors = {"Low": "#54A24B", "Moderate": "#EECA3B", "High": "#F58518", "Severe": "#E45756"}
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(dist.index, dist.values * 100, color=[colors[l] for l in dist.index])
for i, v in enumerate(dist.values * 100): ax.text(i, v + 1, f"{v:.1f}%", ha="center")
ax.set(title="Distribusi level kemacetan (semua sensor, semua waktu)", ylabel="% pembacaan", ylim=(0, 100))
savefig("02_distribusi_level"); plt.show()

**Pelajaran penting:** sekitar 9 dari 10 pembacaan berlabel *Low*. Model yang selalu menjawab "Low" sudah dapat akurasi ~90% — tapi **tidak berguna** bagi city planner. Karena itu nanti kita menilai model dengan **macro-F1** dan **recall per kelas**, bukan akurasi.

## 4. Master dataset (prinsip inti Lambda Architecture)
Di Lambda Architecture, **master dataset** adalah sumber kebenaran tunggal:
- **Immutable** — tidak pernah diedit; data baru hanya *ditambahkan* (append).
- Berisi **fakta mentah** (timestamp, sensor, kecepatan). Hal turunan seperti rasio & level dihitung belakangan sebagai *view*, sehingga kalau aturannya berubah (mis. ambang level), kita cukup menghitung ulang view tanpa merusak data asli.

Kita ubah format **wide → long** (satu baris = satu pembacaan) karena format long yang dipakai Spark dan database, lalu simpan sebagai **Parquet terpartisi per bulan**.

**Kenapa Parquet?** Kolumnar + terkompresi → jauh lebih kecil dari CSV dan Spark hanya membaca kolom yang dibutuhkan. **Kenapa partisi per bulan?** Query "data Maret saja" cukup membaca satu folder (*partition pruning*).

In [ ]:
T, S = df.shape
master = pd.DataFrame({
    "timestamp":  np.repeat(df.index.values.astype("datetime64[us]"), S),
    "sensor_id":  np.tile(df.columns.values.astype("int32"), T),
    "speed":      df.values.ravel(),
    "is_imputed": is_imputed.values.ravel(),
}).dropna(subset=["speed"])
master["month"] = master["timestamp"].dt.month.astype("int8")

if os.path.exists(MASTER_DIR): shutil.rmtree(MASTER_DIR)
pq.write_to_dataset(pa.Table.from_pandas(master, preserve_index=False), MASTER_DIR, partition_cols=["month"])

size_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(MASTER_DIR) for f in fs) / 1e6
print(f"Master dataset: {len(master):,} baris | Parquet {size_mb:.0f} MB (CSV asli 86 MB)")
print("Partisi:", sorted(os.listdir(MASTER_DIR)))
master.head()

In [ ]:
# Tabel referensi kecil — disimpan terpisah dari master dataset
v_ff.reset_index().to_csv(f"{OUT_DIR}/ref_free_flow_speed.csv", index=False)
meta.to_csv(f"{OUT_DIR}/ref_sensor_meta.csv", index=False)
del master

## 5. EDA — pola kemacetan
Pertanyaan yang ingin dijawab seorang city planner:
1. Bagaimana tren kecepatan jaringan dari waktu ke waktu?
2. **Jam & hari apa** paling macet?
3. **Di mana** titik-titik macetnya (hotspot)?
4. Seberapa besar kemacetan itu **rutin** (bisa ditebak dari jadwal) vs **tidak rutin** (insiden, cuaca)?

In [ ]:
hourly = df.mean(axis=1).resample("1h").mean()
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(hourly.index, hourly.values, lw=0.5, color="#4C78A8")
ax.plot(hourly.rolling(24 * 7).mean(), lw=2, color="#E45756", label="rata-rata 7 hari")
ax.set(title="Kecepatan rata-rata jaringan per jam (Jan–Jun 2017)", ylabel="mph"); ax.legend()
savefig("03_tren_kecepatan_jaringan"); plt.show()

**Perhatikan:** turunan tajam berulang setiap hari kerja (jam sibuk), lebih landai di akhir pekan. Turunan yang lebih dalam dari biasanya = kejadian tidak rutin.

In [ ]:
DAYS = ["Sen", "Sel", "Rab", "Kam", "Jum", "Sab", "Min"]
congested = (ratio < 0.6).astype("float32").where(ratio.notna())        # True bila High/Severe
pct = congested.mean(axis=1)                          # % sensor macet tiap 5 menit
heat = pct.groupby([pct.index.dayofweek, pct.index.hour]).mean().unstack() * 100

fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(7), DAYS); ax.set_xticks(range(24)); ax.set_xlabel("Jam")
ax.set_title("% sensor dalam kondisi High/Severe — hari × jam"); ax.grid(False)
plt.colorbar(im, label="%"); savefig("04_heatmap_hari_jam"); plt.show()
print("Puncak:", DAYS[heat.stack().idxmax()[0]], f"jam {heat.stack().idxmax()[1]}:00 →", f"{heat.values.max():.1f}% sensor macet")

In [ ]:
lvl_by_hour = pd.DataFrame({
    lv: ((ratio >= lo) & (ratio < hi)).astype("float32").where(ratio.notna()).mean(axis=1)
          .groupby(lambda t: t.hour).mean() * 100
    for lv, lo, hi in [("Moderate", .6, .8), ("High", .4, .6), ("Severe", 0, .4)]
})
ax = lvl_by_hour.plot(kind="bar", stacked=True, figsize=(11, 3), color=[colors[c] for c in lvl_by_hour.columns], width=0.85)
ax.set(title="Komposisi kondisi tidak lancar per jam (semua hari)", xlabel="Jam", ylabel="% pembacaan")
savefig("05_level_per_jam"); plt.show()

### Hotspot: sensor mana yang paling sering macet?

In [ ]:
hot = pd.DataFrame({
    "pct_congested": congested.mean() * 100,
    "pct_severe":    (ratio < 0.4).astype("float32").where(ratio.notna()).mean() * 100,
    "avg_speed":     df.mean(),
}).rename_axis("sensor_id").reset_index().merge(meta, on="sensor_id")

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(hot.longitude, hot.latitude, c=hot.pct_congested, s=12 + hot.pct_congested * 4, cmap="YlOrRd", edgecolor="k", lw=0.3)
plt.colorbar(sc, label="% waktu High/Severe")
ax.set(title="Peta hotspot kemacetan — 325 sensor PEMS-BAY", xlabel="Longitude", ylabel="Latitude")
savefig("06_peta_hotspot"); plt.show()
hot.sort_values("pct_congested", ascending=False).head(10)[["sensor_id", "fwy", "dir", "name", "lanes", "pct_congested", "pct_severe", "avg_speed"]].round(1)

### Seberapa rutin kemacetan ini?
Kita uji **baseline paling sederhana**: tebak kecepatan = rata-rata historis sensor itu pada *hari & jam yang sama*. Ukur dengan **R²** (proporsi variasi yang bisa dijelaskan).

- Kalau R² ≈ 1 → kemacetan sepenuhnya rutin, ML tidak banyak berguna (ini masalah dataset sintetis kita sebelumnya: ~95%).
- Kalau R² jauh dari 1 → ada bagian **tidak rutin** yang perlu ditangkap model dari data terkini (lag, tren).

In [ ]:
how = df.index.dayofweek * 24 + df.index.hour           # hour-of-week: 0..167
profile_pred = df.groupby(how).transform("mean")
ok = df.notna().values
ss_res = np.nansum((df.values - profile_pred.values)[ok] ** 2)
ss_tot = np.nansum((df.values - np.nanmean(df.values))[ok] ** 2)
r2 = 1 - ss_res / ss_tot
print(f"R² baseline 'sensor × hari × jam' = {r2:.3f}")
print(f"→ {r2:.0%} variasi kecepatan bersifat rutin; {1 - r2:.0%} tidak rutin. Inilah ruang kerja Model B.")
del profile_pred

## 6. Batch Layer dengan PySpark
Sekarang bagian Big Data-nya. Pandas cukup untuk 17 juta baris di satu mesin, tapi kalau datanya ribuan sensor × bertahun-tahun, kita butuh **komputasi terdistribusi**. Spark membagi data ke banyak *partition* dan memprosesnya paralel — kode yang sama berjalan di laptop maupun cluster 100 mesin.

**Batch view** = tabel hasil pra-komputasi dari master dataset, siap dibaca cepat oleh serving layer (dashboard). Kita buat 4:

| Batch view | Grain | Dipakai untuk |
|---|---|---|
| `bv_sensor_hourly` | sensor × tanggal × jam | grafik historis per lokasi |
| `bv_sensor_profile` | sensor × hari × jam | "kondisi normal" pembanding real-time & baseline Model B |
| `bv_network_daily` | tanggal | KPI kota harian |
| `bv_hotspots` | sensor | peringkat lokasi rawan macet + peta |

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder
         .master("local[*]")                              # pakai semua core CPU
         .appName("BatchLayer-PEMSBAY")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.session.timeZone", "UTC")     # cegah pergeseran jam
         .config("spark.sql.shuffle.partitions", "16")    # data kecil → partisi sedikit lebih cepat
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
master_sdf = spark.read.parquet(MASTER_DIR)
vff_sdf    = spark.createDataFrame(v_ff.reset_index())
meta_sdf   = spark.createDataFrame(meta)

master_sdf.printSchema()
print("Jumlah baris master:", f"{master_sdf.count():,}")
print("Jumlah partisi Spark:", master_sdf.rdd.getNumPartitions())

### 6.1 Enrichment: tambahkan rasio, level, dan atribut waktu
Perhatikan `F.broadcast(...)`: tabel v_ff hanya 325 baris, jadi Spark mengirim salinannya ke semua worker alih-alih mengacak (*shuffle*) 17 juta baris. Ini optimasi join yang sangat umum di Big Data.

In [ ]:
ratio_col = F.least(F.col("speed") / F.col("v_ff"), F.lit(1.5))

enriched = (master_sdf
    .join(F.broadcast(vff_sdf), "sensor_id")
    .withColumn("speed_ratio", ratio_col)
    .withColumn("level", F.when(F.col("speed_ratio") < 0.4, "Severe")
                          .when(F.col("speed_ratio") < 0.6, "High")
                          .when(F.col("speed_ratio") < 0.8, "Moderate")
                          .otherwise("Low"))
    .withColumn("is_congested", (F.col("speed_ratio") < 0.6).cast("int"))
    .withColumn("is_severe",    (F.col("speed_ratio") < 0.4).cast("int"))
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("dow",  ((F.dayofweek("timestamp") + 5) % 7))   # 0 = Senin … 6 = Minggu
    .cache())                                                    # dipakai berkali-kali → simpan di memori

enriched.select("timestamp", "sensor_id", "speed", "v_ff", "speed_ratio", "level", "dow", "hour").show(5)
enriched.groupBy("level").count().orderBy("count", ascending=False).show()

### 6.2 Menghitung batch views

In [ ]:
bv_sensor_hourly = (enriched.groupBy("sensor_id", "date", "hour").agg(
    F.round(F.avg("speed"), 2).alias("avg_speed"),
    F.round(F.min("speed"), 2).alias("min_speed"),
    F.round(F.avg("speed_ratio"), 3).alias("avg_ratio"),
    F.round(F.avg("is_congested"), 3).alias("pct_congested"),
    F.count("*").alias("n_obs")))

bv_sensor_profile = (enriched.groupBy("sensor_id", "dow", "hour").agg(
    F.round(F.avg("speed"), 2).alias("mean_speed"),
    F.round(F.percentile_approx("speed", 0.15), 2).alias("p15_speed"),
    F.round(F.avg("is_congested"), 3).alias("prob_congested"),
    F.round(F.avg("is_severe"), 3).alias("prob_severe"),
    F.count("*").alias("n_obs")))

bv_network_daily = (enriched.groupBy("date").agg(
    F.round(F.avg("speed"), 2).alias("avg_speed"),
    F.round(F.avg("is_congested") * 100, 2).alias("pct_congested"),
    F.countDistinct(F.when(F.col("is_severe") == 1, F.col("sensor_id"))).alias("n_sensor_severe"))
    .withColumn("dow", (F.dayofweek("date") + 5) % 7)
    .orderBy("date"))

w = Window.orderBy(F.desc("pct_congested"))
bv_hotspots = (enriched.groupBy("sensor_id").agg(
    F.round(F.avg("is_congested") * 100, 2).alias("pct_congested"),
    F.round(F.avg("is_severe") * 100, 2).alias("pct_severe"),
    F.round(F.avg("speed"), 2).alias("avg_speed"),
    F.first("v_ff").alias("v_ff"))
    .join(F.broadcast(meta_sdf), "sensor_id")
    .withColumn("rank", F.row_number().over(w))
    .orderBy("rank"))

for name, sdf in [("bv_sensor_hourly", bv_sensor_hourly), ("bv_sensor_profile", bv_sensor_profile),
                  ("bv_network_daily", bv_network_daily), ("bv_hotspots", bv_hotspots)]:
    print(f"{name:18s} → {sdf.count():>9,} baris")

In [ ]:
bv_hotspots.select("rank", "sensor_id", "fwy", "dir", "name", "pct_congested", "pct_severe", "avg_speed").show(10, truncate=False)

### 6.3 Spark SQL — menjawab pertanyaan city planner
Batch view bisa di-query dengan SQL biasa. Ini cara analis non-programmer memakai hasil batch layer.

In [ ]:
bv_sensor_profile.createOrReplaceTempView("profile")
meta_sdf.createOrReplaceTempView("sensor")

# Q1: Di hari kerja, jam berapa jaringan paling sering macet?
spark.sql('''
    SELECT hour, ROUND(AVG(prob_congested) * 100, 2) AS pct_congested
    FROM profile WHERE dow < 5
    GROUP BY hour ORDER BY pct_congested DESC LIMIT 5
''').show()

# Q2: Jalan tol (fwy + arah) mana yang paling bermasalah pada jam sibuk sore (16–18)?
spark.sql('''
    SELECT s.fwy, s.dir, COUNT(DISTINCT p.sensor_id) AS n_sensor,
           ROUND(AVG(p.prob_congested) * 100, 2) AS pct_congested
    FROM profile p JOIN sensor s USING (sensor_id)
    WHERE p.dow < 5 AND p.hour BETWEEN 16 AND 18
    GROUP BY s.fwy, s.dir ORDER BY pct_congested DESC LIMIT 5
''').show()

### 6.4 Simpan batch views
Setiap view disimpan sebagai Parquet (untuk Spark/model) + CSV untuk view kecil (mudah dibaca backend dashboard). Lalu semua disalin ke Google Drive.

In [ ]:
BV_LOCAL = f"{WORK}/batch_views"
views = {"bv_sensor_hourly": bv_sensor_hourly, "bv_sensor_profile": bv_sensor_profile,
         "bv_network_daily": bv_network_daily, "bv_hotspots": bv_hotspots}

for name, sdf in views.items():
    sdf.coalesce(1).write.mode("overwrite").parquet(f"{BV_LOCAL}/{name}")
for name in ["bv_sensor_profile", "bv_network_daily", "bv_hotspots"]:      # yang kecil → CSV juga
    views[name].toPandas().to_csv(f"{OUT_DIR}/{name}.csv", index=False)

# salin ke Drive (hasil Spark + master dataset untuk notebook 04)
for src, dst in [(BV_LOCAL, f"{OUT_DIR}/batch_views"), (MASTER_DIR, f"{OUT_DIR}/master_pemsbay")]:
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst)

enriched.unpersist()
print("Tersimpan di", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)

## 7. Ringkasan
| Hasil | Lokasi |
|---|---|
| Master dataset (Parquet per bulan) | `output/01_batch/master_pemsbay/` |
| 4 batch views (Parquet) | `output/01_batch/batch_views/` |
| Batch views kecil (CSV, untuk dashboard) | `output/01_batch/*.csv` |
| Tabel referensi v_ff & metadata sensor | `output/01_batch/ref_*.csv` |
| Grafik untuk laporan | `output/01_batch/figures/` |

**Temuan kunci untuk laporan:**
1. Data nyata punya cacat kecil (timestamp hilang, nilai 0) yang perlu ditangani eksplisit.
2. Kemacetan jarang (~9% pembacaan) → evaluasi model wajib pakai macro-F1/recall, bukan akurasi.
3. Kemacetan terkonsentrasi di jam sibuk hari kerja dan di sejumlah kecil hotspot.
4. Pola rutin menjelaskan sebagian besar variasi, tapi tidak semuanya → Model B harus mengalahkan baseline profil `bv_sensor_profile`.

---
## 8. Latihan mandiri (kerjakan sendiri dulu!)
Inilah bagian yang membuatmu benar-benar bisa. Tidak ada jawaban di notebook ini — coba sendiri, lalu tunjukkan hasilnya ke saya untuk direview.

**Latihan 1 — Ubah ambang level.** Ganti `BINS` menjadi `[0, 0.5, 0.7, 0.85, np.inf]`, hitung ulang distribusi level. Apa dampaknya bagi keseimbangan kelas? Ambang mana yang lebih masuk akal untuk jalan kota Indonesia, dan kenapa?

*Petunjuk:* salin kode di bagian 3, cukup ubah satu baris.

In [ ]:
# TODO Latihan 1


**Latihan 2 — Spark SQL.** Tulis query ke view `profile`: tampilkan **5 sensor** dengan `prob_severe` tertinggi pada **hari Senin jam 08:00**, lengkap dengan nama jalannya (join ke `sensor`).

*Petunjuk:* Senin = `dow = 0`. Pola query-nya mirip Q2.

In [ ]:
# TODO Latihan 2


**Latihan 3 — Batch view baru.** Buat `bv_freeway_hourly`: rata-rata kecepatan dan % macet **per jalan tol (fwy) per jam**, khusus hari kerja. Simpan sebagai CSV.

*Petunjuk:* `enriched` sudah di-unpersist; buat ulang dari `master_sdf` atau jalankan ulang sel 6.1. Kamu perlu join ke `meta_sdf` untuk mendapatkan `fwy`, lalu `groupBy("fwy", "hour")`.

In [ ]:
# TODO Latihan 3


---
**Selanjutnya → Notebook 02:** fine-tuning YOLOv8 dengan dataset Roboflow untuk mendeteksi kendaraan di video (fondasi speed layer).